# Análisis exploratorio de transacciones

Este notebook es únicamente para análisis exploratorio. No implementa la arquitectura Medallón ni transforma la fuente: Airflow ejecuta la ETL y Hadoop/HDFS guarda sus capas como directorios. Aquí se leen las salidas publicadas por Airflow en el volumen compartido:

- **Bronze:** `/bancomnm/bronze/transacciones/sha256=<hash>/dataset.json` — copia cruda en HDFS.
- **Silver:** `/bancomnm/silver/transacciones/dataset.jsonl` — datos validados y estandarizados.
- **Gold:** `/bancomnm/gold/job*/` — resultados de los jobs MapReduce.

El DAG de Airflow publica Silver y Gold en `/data` para facilitar este análisis. El escaneo de Silver es por fragmentos para no cargar el millón de filas completo en memoria.

## Requisitos

Primero ejecuta el DAG `bancomnm_mapreduce_pipeline` en Airflow y espera a que termine en estado **success**. El servicio opcional de Jupyter monta `airflow_data` en `/data`, que es donde el DAG publica Silver y Gold.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_DIR = Path("/data")
SILVER_PATH = DATA_DIR / "silver" / "transacciones.jsonl"
QUALITY_PATH = DATA_DIR / "silver" / "quality_report.json"
GOLD_DIR = DATA_DIR / "gold"

if not SILVER_PATH.is_file():
    raise FileNotFoundError(
        "No existe Silver todavía. Ejecuta el DAG bancomnm_mapreduce_pipeline "
        "y vuelve a abrir este notebook cuando finalice."
    )
if not QUALITY_PATH.is_file():
    raise FileNotFoundError("Falta el reporte de calidad publicado por Airflow.")

print(f"Silver: {SILVER_PATH}")
print(f"Gold:   {GOLD_DIR}")

## Calidad de datos en Silver

El DAG comprueba campos requeridos, tipos y rangos, fechas, estados permitidos e identificadores duplicados. Si encuentra registros inválidos, los pone en cuarentena y detiene la publicación de Silver.

In [ ]:
quality_report = json.loads(QUALITY_PATH.read_text(encoding="utf-8"))
display(pd.DataFrame([quality_report]).T.rename(columns={0: "valor"}))

quarantine_path = DATA_DIR / "silver" / "quarantine.jsonl"
if quarantine_path.is_file() and quarantine_path.stat().st_size:
    print(f"Hay registros rechazados en {quarantine_path}")
else:
    print("No hay registros en cuarentena.")

## Perfil de Silver

Se calculan conteos y estadísticas incrementales. Para los gráficos de distribución se conserva una muestra determinista y acotada de hasta 20.000 transacciones.

In [ ]:
rows_scanned = 0
amount_sum = 0.0
amount_min = None
amount_max = None
missing_counts = None
status_counts = {}
fraud_counts = {}
sample_chunks = []

for chunk_number, chunk in enumerate(
    pd.read_json(SILVER_PATH, lines=True, chunksize=100_000, convert_dates=False)
):
    rows_scanned += len(chunk)
    chunk_missing = chunk.isna().sum()
    missing_counts = (
        chunk_missing if missing_counts is None else missing_counts.add(chunk_missing, fill_value=0)
    )

    amounts = pd.to_numeric(chunk["monto_cop"], errors="raise")
    amount_sum += amounts.sum()
    current_min, current_max = amounts.min(), amounts.max()
    amount_min = current_min if amount_min is None else min(amount_min, current_min)
    amount_max = current_max if amount_max is None else max(amount_max, current_max)

    for value, count in chunk["estado"].value_counts().items():
        status_counts[value] = status_counts.get(value, 0) + int(count)
    for value, count in chunk["fraude"].value_counts().items():
        fraud_counts[int(value)] = fraud_counts.get(int(value), 0) + int(count)

    sample_chunks.append(
        chunk.sample(n=min(2_000, len(chunk)), random_state=42 + chunk_number)
    )

sample = pd.concat(sample_chunks, ignore_index=True)
profile = pd.DataFrame(
    {
        "métrica": ["filas", "monto total (COP)", "monto promedio (COP)", "mínimo (COP)", "máximo (COP)"],
        "valor": [rows_scanned, amount_sum, amount_sum / rows_scanned, amount_min, amount_max],
    }
)
display(profile)
display(pd.DataFrame({"nulos": missing_counts.astype("int64")}))
display(pd.Series(status_counts, name="transacciones").sort_index().to_frame())
display(pd.Series(fraud_counts, name="transacciones").sort_index().to_frame())
print(f"Muestra para visualizaciones: {len(sample):,} filas")

## Tablas Gold de MapReduce

Cada job produce uno o más archivos `part-*` TSV, sin encabezado. La función siguiente combina las particiones de cada resultado.

In [ ]:
def read_gold(job_name, columns):
    job_dir = GOLD_DIR / job_name
    part_files = sorted(job_dir.glob("part-*"))
    if not part_files:
        raise FileNotFoundError(
            f"No hay resultados para {job_name}. Confirma que el DAG terminó en success."
        )
    frames = [
        pd.read_csv(part, sep="\t", header=None, names=columns)
        for part in part_files
        if part.stat().st_size
    ]
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True)


gold_category = read_gold("job1_categoria", ["categoria", "total_monto", "count"])
gold_fraud = read_gold("job2_fraude", ["ciudad", "total_fraudes"])
gold_customer = read_gold(
    "job3_clientes", ["customer_id", "total_gastado", "num_transacciones"]
)
gold_monthly = read_gold("job4_mensual", ["mes", "canal", "total_monto", "count"])

display(gold_category.sort_values("total_monto", ascending=False).head(10))
display(gold_fraud.sort_values("total_fraudes", ascending=False).head(10))
display(gold_customer.sort_values("total_gastado", ascending=False).head(10))
display(gold_monthly.sort_values(["mes", "canal"]).head(10))

## Distribuciones de la muestra Silver

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sample["monto_cop"].clip(upper=sample["monto_cop"].quantile(0.99)).hist(
    bins=40, ax=axes[0]
)
axes[0].set_title("Monto por transacción (hasta p99, muestra)")
axes[0].set_xlabel("COP")
sample["estado"].value_counts().sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Estados en la muestra")
axes[1].set_xlabel("Transacciones")
plt.tight_layout()
plt.show()

## Indicadores Gold

Estos gráficos usan los agregados distribuidos completos, no solo la muestra del perfil Silver.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
gold_category.sort_values("total_monto").plot.barh(
    x="categoria", y="total_monto", legend=False, ax=axes[0]
)
axes[0].set_title("Monto aprobado por categoría — Gold")
axes[0].set_xlabel("COP")

monthly_pivot = gold_monthly.pivot_table(
    index="mes", columns="canal", values="total_monto", aggfunc="sum"
).sort_index()
monthly_pivot.plot(ax=axes[1], marker="o")
axes[1].set_title("Monto por mes y canal — Gold")
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("COP")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

top_customers = gold_customer.nlargest(10, "total_gastado").sort_values("total_gastado")
top_customers.plot.barh(
    x="customer_id", y="total_gastado", legend=False, figsize=(8, 5)
)
plt.title("10 clientes con mayor gasto aprobado — Gold")
plt.xlabel("COP")
plt.tight_layout()
plt.show()

gold_fraud.nlargest(10, "total_fraudes").sort_values("total_fraudes").plot.barh(
    x="ciudad", y="total_fraudes", legend=False, figsize=(8, 5)
)
plt.title("Fraudes por ciudad — Gold")
plt.xlabel("Transacciones marcadas como fraude")
plt.tight_layout()
plt.show()

## Preguntas para el análisis

- ¿Qué categorías y canales concentran mayor monto aprobado?
- ¿Qué ciudades presentan más transacciones marcadas como fraude?
- ¿Cómo se compara el gasto de los clientes con el volumen total?
- ¿Qué validaciones adicionales convendría incorporar al contrato de Silver?

Las conclusiones deben basarse en las tablas y gráficos generados en la ejecución del notebook.